In [ ]:
import pandas as pd
import scanpy as sc
from scipy import io
from scipy.sparse import csr_matrix
from pathlib import Path
import numpy as np
import scvi
import celltypist
from celltypist import models

import torch
# torch.set_float32_matmul_precision("high")
# torch.set_num_threads(31)

In [ ]:
############################
# Import dataset from R QC #
############################

base = Path("../data/001_Integrated_export")

def load_mod(mod):
    mdir = base / mod
    feats = pd.read_csv(mdir / "features.tsv", sep="\t")["feature"].astype(str).values
    bcs   = pd.read_csv(mdir / "barcodes.tsv", sep="\t", header=None)[0].astype(str).values
    X_data   = io.mmread(mdir / "matrix_data.mtx").tocsr().T
    X_counts = io.mmread(mdir / "matrix_counts.mtx").tocsr().T
    ad = sc.AnnData(X_data)
    ad.layers["counts"] = X_counts
    ad.var_names = pd.Index(feats)
    ad.obs_names = pd.Index(bcs)
    return ad

def load_with_obs():
    rna = load_mod("rna")
    obs = pd.read_csv(base / "obs.csv", index_col=0)
    common = obs.index.intersection(rna.obs_names)
    if len(common) >= 0.9 * rna.n_obs:
        rna = rna[common, :].copy()
        rna.obs = obs.loc[common].copy()
        return rna

    obs_idx = pd.Index(obs.index.astype(str))
    if obs_idx.str.contains("_").any():
        raw = obs_idx.str.split("_", 1).str[1]
        mapper = pd.Series(obs_idx, index=raw)
        keep_mask = pd.Index(rna.obs_names).isin(mapper.index)
        if keep_mask.sum() == 0:
            raise ValueError("No overlap between rna.obs_names and obs indices. Check export.")
        rna.obs_names = pd.Index(mapper.loc[rna.obs_names].values, name=None)
        # now exact alignment
        rna = rna[obs.index.intersection(rna.obs_names), :].copy()
        rna.obs = obs.loc[rna.obs_names].copy()
        rna.obs_names_make_unique()
        return rna
    raise ValueError(
        f"Low overlap: {len(common)} of {rna.n_obs}. "
        "Likely barcode prefix mismatch. Ensure Seurat cell names were renamed BEFORE export."
    )

adata = load_with_obs()
print(adata)


In [ ]:
sc.pp.highly_variable_genes(
    adata, n_top_genes=2000, flavor="cell_ranger", batch_key="sample"
)

adata_scvi = adata[:, adata.var["highly_variable"]].copy()

In [ ]:
scvi.model.SCVI.setup_anndata(adata_scvi, layer="counts", batch_key="sample")
model_scvi = scvi.model.SCVI(adata_scvi)
model_scvi.view_anndata_setup()

In [ ]:

max_epochs_scvi = np.min([round((20000 / adata.n_obs) * 400), 400])

model_scvi.train(
    accelerator="gpu",
    devices=1,
    batch_size=128,                 # <- keep this ONLY here
    datasplitter_kwargs={
        "num_workers": 31,          # dataloader workers
        "pin_memory": True,         # faster H2D copies
        "persistent_workers": True  # keep workers alive across epochs
    },
)

In [ ]:
adata_scvi.obsm["X_scVI"] = model_scvi.get_latent_representation()

if (adata_scvi.obs_names == adata.obs_names).all():
    adata.obsm["X_scVI"] = adata_scvi.obsm["X_scVI"]
else:
    idx = adata.obs_names.get_indexer(adata_scvi.obs_names)
    X = adata_scvi.obsm["X_scVI"]
    X_full = np.zeros((adata.n_obs, X.shape[1]), dtype=X.dtype)
    X_full[idx, :] = X
    adata.obsm["X_scVI"] = X_full

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.pl.umap(adata, color="sample", wspace=1)

In [ ]:
adata.X = adata.layers["counts"]
sc.pp.normalize_total(adata, target_sum=10**4)
sc.pp.log1p(adata)
adata.X = adata.X.toarray()


models.download_models(
    force_update=True,
    model=['Developing_Human_Brain.pkl']
)
Dev_Human_Brain = models.Model.load(model="Developing_Human_Brain.pkl")

predictions=celltypist.annotate(
    adata, model= Dev_Human_Brain, majority_voting=True
)
predictions_adata =  predictions.to_adata()
adata.obs["celltypist_cell_label"] = predictions_adata.obs.loc[
    adata.obs.index, "majority_voting"
]
adata.obs["celltypist_conf_score"] = predictions_adata.obs.loc[
    adata.obs.index, "conf_score"
]

sc.pl.umap(
    adata,
    color=["celltypist_cell_label", "celltypist_conf_score"],
    frameon=False,
    sort_order=False,
    wspace=1,
)

adata.write_h5ad("../data/002_Integrated.h5ad")

In [ ]:
adata=sc.read_h5ad("../data/002_Integrated.h5ad")